# LogStar — Qwen3 Embedding: 12 ca so với E5

Chỉ chạy Qwen3-Embedding-0.6B **một lượt trên Kaggle CPU**, không train, không chạy inference local.
Giữ nguyên 31 tên sản phẩm, 12 truy vấn và nhãn từ phép thử E5: 10 ca có nhãn, RD6 và Chuối Cavendish chỉ xem gợi ý.

So với kết quả E5 đã chạy ở Version 1 trước đó, không chạy lại E5. Dữ liệu tổng hợp nhỏ, không đại diện accuracy thực tế. Chỉ dẫn được đặt trước lượt Qwen này nhưng có tham khảo lỗi E5 đã biết; đây không phải tập test độc lập.
Không tách query bằng LLM khác hoặc luật giống, không alias enrichment, không lọc loại trước. Toàn bộ query + instruction đi thẳng vào Qwen để tạo vector.


In [ ]:
import os
assert os.path.isdir('/kaggle/working'), 'Chạy notebook này trên Kaggle; không chạy inference local.'
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'sentence-transformers==5.7.0', 'transformers==5.19.0'])
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


In [ ]:
import gc, json, time, platform, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from IPython.display import display

torch.set_num_threads(2)
torch.manual_seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Kaggle device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
OUT = Path('/kaggle/working/qwen-e5-tiny-results')
OUT.mkdir(exist_ok=True)


In [ ]:
PAYLOAD = json.loads('{"catalog": [{"id": "durian-0", "group": "durian", "fruit": "Sầu riêng", "variety": "Ri6", "name": "Sầu riêng Ri6", "aliases": ["Ri 6", "RI-6"]}, {"id": "durian-1", "group": "durian", "fruit": "Sầu riêng", "variety": "Monthong", "name": "Sầu riêng Monthong", "aliases": ["Dona", "Đô Na", "Mon Thong"]}, {"id": "durian-2", "group": "durian", "fruit": "Sầu riêng", "variety": "Musang King", "name": "Sầu riêng Musang King", "aliases": ["Musangking"]}, {"id": "durian-3", "group": "durian", "fruit": "Sầu riêng", "variety": "Black Thorn", "name": "Sầu riêng Black Thorn", "aliases": ["Blackthorn"]}, {"id": "durian-4", "group": "durian", "fruit": "Sầu riêng", "variety": "Chín Hóa", "name": "Sầu riêng Chín Hóa", "aliases": []}, {"id": "durian-5", "group": "durian", "fruit": "Sầu riêng", "variety": "Khổ qua xanh", "name": "Sầu riêng Khổ qua xanh", "aliases": []}, {"id": "durian-6", "group": "durian", "fruit": "Sầu riêng", "variety": "Chuồng bò", "name": "Sầu riêng Chuồng bò", "aliases": []}, {"id": "mango-0", "group": "mango", "fruit": "Xoài", "variety": "Cát Hòa Lộc", "name": "Xoài Cát Hòa Lộc", "aliases": []}, {"id": "mango-1", "group": "mango", "fruit": "Xoài", "variety": "Cát Chu", "name": "Xoài Cát Chu", "aliases": []}, {"id": "mango-2", "group": "mango", "fruit": "Xoài", "variety": "Keo", "name": "Xoài Keo", "aliases": []}, {"id": "mango-3", "group": "mango", "fruit": "Xoài", "variety": "Tượng da xanh", "name": "Xoài Tượng da xanh", "aliases": []}, {"id": "pomelo-0", "group": "pomelo", "fruit": "Bưởi", "variety": "Da xanh", "name": "Bưởi Da xanh", "aliases": []}, {"id": "pomelo-1", "group": "pomelo", "fruit": "Bưởi", "variety": "Năm Roi", "name": "Bưởi Năm Roi", "aliases": []}, {"id": "pomelo-2", "group": "pomelo", "fruit": "Bưởi", "variety": "Diễn", "name": "Bưởi Diễn", "aliases": []}, {"id": "pomelo-3", "group": "pomelo", "fruit": "Bưởi", "variety": "Đoan Hùng", "name": "Bưởi Đoan Hùng", "aliases": []}, {"id": "jackfruit-0", "group": "jackfruit", "fruit": "Mít", "variety": "Thái", "name": "Mít Thái", "aliases": []}, {"id": "jackfruit-1", "group": "jackfruit", "fruit": "Mít", "variety": "Ruột đỏ", "name": "Mít Ruột đỏ", "aliases": []}, {"id": "jackfruit-2", "group": "jackfruit", "fruit": "Mít", "variety": "Tố nữ", "name": "Mít Tố nữ", "aliases": []}, {"id": "longan-0", "group": "longan", "fruit": "Nhãn", "variety": "Xuồng cơm vàng", "name": "Nhãn Xuồng cơm vàng", "aliases": []}, {"id": "longan-1", "group": "longan", "fruit": "Nhãn", "variety": "Tiêu da bò", "name": "Nhãn Tiêu da bò", "aliases": []}, {"id": "longan-2", "group": "longan", "fruit": "Nhãn", "variety": "Ido", "name": "Nhãn Ido", "aliases": []}, {"id": "dragonfruit-0", "group": "dragonfruit", "fruit": "Thanh long", "variety": "Ruột trắng", "name": "Thanh long Ruột trắng", "aliases": []}, {"id": "dragonfruit-1", "group": "dragonfruit", "fruit": "Thanh long", "variety": "Ruột đỏ LĐ1", "name": "Thanh long Ruột đỏ LĐ1", "aliases": ["Long Định 1", "LĐ1", "LD1"]}, {"id": "dragonfruit-2", "group": "dragonfruit", "fruit": "Thanh long", "variety": "Ruột tím hồng LĐ5", "name": "Thanh long Ruột tím hồng LĐ5", "aliases": ["Long Định 5", "LĐ5", "LD5"]}, {"id": "rice-0", "group": "rice", "fruit": "Lúa", "variety": "ST24", "name": "Lúa ST24", "aliases": []}, {"id": "rice-1", "group": "rice", "fruit": "Lúa", "variety": "ST25", "name": "Lúa ST25", "aliases": []}, {"id": "rice-2", "group": "rice", "fruit": "Lúa", "variety": "OM5451", "name": "Lúa OM5451", "aliases": []}, {"id": "rice-3", "group": "rice", "fruit": "Lúa", "variety": "OM18", "name": "Lúa OM18", "aliases": []}, {"id": "rice-4", "group": "rice", "fruit": "Lúa", "variety": "IR50404", "name": "Lúa IR50404", "aliases": []}, {"id": "grape-0", "group": "grape", "fruit": "Nho", "variety": "NH01-48", "name": "Nho NH01-48", "aliases": []}, {"id": "grape-1", "group": "grape", "fruit": "Nho", "variety": "NH01-152", "name": "Nho NH01-152", "aliases": []}], "queries": [{"id": "q001", "text": "Sầu riêng Ri6", "target": "durian-0", "category": "exact", "note": ""}, {"id": "q156", "text": "Sầu riêng RD6", "target": null, "category": "ambiguous", "note": "Không có một đáp án duy nhất; chỉ quan sát top/margin, không tính accuracy."}, {"id": "q140", "text": "Sầu riêng Dona", "target": "durian-1", "category": "alias", "note": "Alias trong fixture; thí nghiệm enriched cung cấp alias này cho passage một cách công khai."}, {"id": "q002", "text": "sau rieng ri6", "target": "durian-0", "category": "unaccented", "note": ""}, {"id": "q009", "text": "Sầu riêng Musang King", "target": "durian-2", "category": "exact", "note": ""}, {"id": "q148", "text": "Cần Ri6, không mua Monthong", "target": "durian-0", "category": "hard_context", "note": ""}, {"id": "q033", "text": "Xoài Cát Chu", "target": "mango-1", "category": "exact", "note": ""}, {"id": "q152", "text": "Xoài Cát Chu, không lấy Cát Hòa Lộc", "target": "mango-1", "category": "hard_context", "note": ""}, {"id": "q049", "text": "Bưởi Năm Roi", "target": "pomelo-1", "category": "exact", "note": ""}, {"id": "q101", "text": "Lúa ST25", "target": "rice-1", "category": "exact", "note": ""}, {"id": "q154", "text": "Gạo ST25, không phải ST24", "target": "rice-1", "category": "hard_context", "note": ""}, {"id": "q164", "text": "Chuối Cavendish", "target": null, "category": "out_of_catalog", "note": "Không có trong catalog đóng; cosine top cao vẫn không chứng minh có giống phù hợp."}], "version": "kaggle-tiny-12-v1"}')

catalog, queries = PAYLOAD['catalog'], PAYLOAD['queries']
names = [p['name'] for p in catalog]
print(len(catalog), 'ứng viên;', len(queries), 'truy vấn; ', sum(q['target'] is not None for q in queries), 'ca có nhãn')
display(pd.DataFrame(queries)[['text','target','category']])


In [ ]:
MODEL_SPECS = [
    ('qwen3-embedding-0.6b', 'Qwen/Qwen3-Embedding-0.6B', '97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3'),
]
TASK = 'Given a Vietnamese agricultural purchase query, retrieve product names matching the requested fruit and cultivar. Prioritize the requested cultivar and avoid cultivars explicitly excluded by the buyer.'
QUERY_PROMPT = 'Instruct: '+TASK+'\nQuery:'
print('Fixed instruction:', TASK)
all_rows, summaries, score_rows = [], [], []
for label, model_id, revision in MODEL_SPECS:
    print('Loading', model_id, flush=True)
    started = time.perf_counter()
    model = SentenceTransformer(model_id, revision=revision, device=DEVICE, trust_remote_code=False, tokenizer_kwargs={'padding_side': 'left'}, model_kwargs={'torch_dtype': torch.float32})
    model.max_seq_length = 512
    loaded = time.perf_counter()-started
    if DEVICE == 'cuda': torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        passages = model.encode(names, prompt='', batch_size=8,
                                normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
        vectors = model.encode([q['text'] for q in queries], prompt=QUERY_PROMPT, batch_size=8,
                               normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    if DEVICE == 'cuda': torch.cuda.synchronize()
    encode_seconds = time.perf_counter()-started
    cosine = vectors @ passages.T
    assert np.isfinite(cosine).all()
    assert vectors.shape == (12, 1024)
    assert np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-5)
    hit1, hit3, reciprocal, count = 0, 0, 0., 0
    for q, scores in zip(queries, cosine):
        order = sorted(range(len(catalog)), key=lambda i: (-float(scores[i]), catalog[i]['id']))
        target_index = next((i for i,p in enumerate(catalog) if p['id']==q['target']), None)
        rank = order.index(target_index)+1 if target_index is not None else None
        if rank is not None:
            count+=1; hit1+=int(rank==1); hit3+=int(rank<=3); reciprocal+=1/rank
        top = order[:3]
        all_rows.append({'model': label, 'query': q['text'], 'category': q['category'],
                         'expected': names[target_index] if target_index is not None else 'Không gán đích',
                         'target_rank': rank, 'top1': names[top[0]], 'cosine_top1': float(scores[top[0]]),
                         'top2': names[top[1]], 'cosine_top2': float(scores[top[1]]),
                         'top3': names[top[2]], 'cosine_top3': float(scores[top[2]]),
                         'top1_top2_margin': float(scores[top[0]]-scores[top[1]])})
        for position, i in enumerate(order, 1):
            score_rows.append({'model': label, 'query': q['text'], 'target': q['target'],
                               'rank': position, 'candidate': names[i], 'cosine': float(scores[i])})
    summaries.append({'model': label, 'model_id': model_id, 'revision': revision,
                      'labelled_queries': count, 'top1_correct': hit1, 'top1_accuracy': hit1/count,
                      'hit_at_3': hit3/count, 'mrr': reciprocal/count,
                      'device': DEVICE, 'model_load_seconds': loaded, 'encode_all_seconds': encode_seconds})
    del model, passages, vectors, cosine
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    print(label, 'finished', flush=True)


In [ ]:
ARCHIVED_E5 = json.loads('{"source": "https://www.kaggle.com/code/tunlnh/logstar-e5-12-variety-cases?scriptVersionId=358645506", "precision": "Values copied from rendered tables (six decimal places).", "tables": [[["", "text", "target", "category"], ["0", "Sầu riêng Ri6", "durian-0", "exact"], ["1", "Sầu riêng RD6", "None", "ambiguous"], ["2", "Sầu riêng Dona", "durian-1", "alias"], ["3", "sau rieng ri6", "durian-0", "unaccented"], ["4", "Sầu riêng Musang King", "durian-2", "exact"], ["5", "Cần Ri6, không mua Monthong", "durian-0", "hard_context"], ["6", "Xoài Cát Chu", "mango-1", "exact"], ["7", "Xoài Cát Chu, không lấy Cát Hòa Lộc", "mango-1", "hard_context"], ["8", "Bưởi Năm Roi", "pomelo-1", "exact"], ["9", "Lúa ST25", "rice-1", "exact"], ["10", "Gạo ST25, không phải ST24", "rice-1", "hard_context"], ["11", "Chuối Cavendish", "None", "out_of_catalog"]], [["", "model", "top1_correct", "labelled_queries", "top1_accuracy", "hit_at_3", "mrr", "device", "encode_all_seconds"], ["0", "small", "8", "10", "0.8", "1.0", "0.90", "cpu", "0.563611"], ["1", "base", "9", "10", "0.9", "1.0", "0.95", "cpu", "1.105418"]], [["", "model", "query", "expected", "top1", "cosine_top1", "top2", "cosine_top2", "target_rank"], ["0", "small", "Sầu riêng Ri6", "Sầu riêng Ri6", "Sầu riêng Ri6", "0.946236", "Sầu riêng Khổ qua xanh", "0.892333", "1.0"], ["1", "small", "Sầu riêng RD6", "Không gán đích", "Sầu riêng Ri6", "0.928092", "Sầu riêng Khổ qua xanh", "0.891547", "NaN"], ["2", "small", "Sầu riêng Dona", "Sầu riêng Monthong", "Sầu riêng Monthong", "0.885456", "Sầu riêng Khổ qua xanh", "0.884150", "1.0"], ["3", "small", "sau rieng ri6", "Sầu riêng Ri6", "Sầu riêng Ri6", "0.884257", "Nhãn Tiêu da bò", "0.841326", "1.0"], ["4", "small", "Sầu riêng Musang King", "Sầu riêng Musang King", "Sầu riêng Musang King", "0.943864", "Sầu riêng Monthong", "0.888963", "1.0"], ["5", "small", "Cần Ri6, không mua Monthong", "Sầu riêng Ri6", "Sầu riêng Monthong", "0.876722", "Sầu riêng Ri6", "0.875120", "2.0"], ["6", "small", "Xoài Cát Chu", "Xoài Cát Chu", "Xoài Cát Chu", "0.924259", "Xoài Cát Hòa Lộc", "0.885653", "1.0"], ["7", "small", "Xoài Cát Chu, không lấy Cát Hòa Lộc", "Xoài Cát Chu", "Xoài Cát Hòa Lộc", "0.906973", "Xoài Cát Chu", "0.894269", "2.0"], ["8", "small", "Bưởi Năm Roi", "Bưởi Năm Roi", "Bưởi Năm Roi", "0.940968", "Bưởi Diễn", "0.882158", "1.0"], ["9", "small", "Lúa ST25", "Lúa ST25", "Lúa ST25", "0.964873", "Lúa ST24", "0.934232", "1.0"], ["10", "small", "Gạo ST25, không phải ST24", "Lúa ST25", "Lúa ST25", "0.873021", "Lúa ST24", "0.859288", "1.0"], ["11", "small", "Chuối Cavendish", "Không gán đích", "Xoài Cát Chu", "0.851795", "Xoài Keo", "0.830734", "NaN"], ["12", "base", "Sầu riêng Ri6", "Sầu riêng Ri6", "Sầu riêng Ri6", "0.915612", "Sầu riêng Chín Hóa", "0.861174", "1.0"], ["13", "base", "Sầu riêng RD6", "Không gán đích", "Sầu riêng Ri6", "0.902955", "Sầu riêng Chín Hóa", "0.855951", "NaN"], ["14", "base", "Sầu riêng Dona", "Sầu riêng Monthong", "Sầu riêng Monthong", "0.855917", "Sầu riêng Chín Hóa", "0.851917", "1.0"], ["15", "base", "sau rieng ri6", "Sầu riêng Ri6", "Sầu riêng Ri6", "0.880795", "Sầu riêng Chín Hóa", "0.828277", "1.0"], ["16", "base", "Sầu riêng Musang King", "Sầu riêng Musang King", "Sầu riêng Musang King", "0.917346", "Sầu riêng Monthong", "0.866347", "1.0"], ["17", "base", "Cần Ri6, không mua Monthong", "Sầu riêng Ri6", "Sầu riêng Ri6", "0.861075", "Sầu riêng Monthong", "0.838177", "1.0"], ["18", "base", "Xoài Cát Chu", "Xoài Cát Chu", "Xoài Cát Chu", "0.883517", "Xoài Cát Hòa Lộc", "0.861421", "1.0"], ["19", "base", "Xoài Cát Chu, không lấy Cát Hòa Lộc", "Xoài Cát Chu", "Xoài Cát Hòa Lộc", "0.879088", "Xoài Cát Chu", "0.864590", "2.0"], ["20", "base", "Bưởi Năm Roi", "Bưởi Năm Roi", "Bưởi Năm Roi", "0.912111", "Bưởi Đoan Hùng", "0.858047", "1.0"], ["21", "base", "Lúa ST25", "Lúa ST25", "Lúa ST25", "0.937826", "Lúa ST24", "0.910117", "1.0"], ["22", "base", "Gạo ST25, không phải ST24", "Lúa ST25", "Lúa ST25", "0.847937", "Lúa ST24", "0.845230", "1.0"], ["23", "base", "Chuối Cavendish", "Không gán đích", "Xoài Cát Chu", "0.812839", "Nhãn Xuồng cơm vàng", "0.809454", "NaN"]]]}')

archived_summary = pd.DataFrame(ARCHIVED_E5['tables'][1][1:], columns=ARCHIVED_E5['tables'][1][0]).drop(columns=[''])
for column in ['top1_correct','labelled_queries','top1_accuracy','hit_at_3','mrr','encode_all_seconds']:
    archived_summary[column] = pd.to_numeric(archived_summary[column])
archived_summary['source'] = 'Archived E5 Kaggle Version 1 (not rerun)'
print('Previous E5 source:', ARCHIVED_E5['source'])
summary = pd.DataFrame(summaries)
details = pd.DataFrame(all_rows)
summary['source'] = 'Current Qwen Kaggle run'
comparison = pd.concat([archived_summary, summary], ignore_index=True)
display(comparison[['model','top1_correct','labelled_queries','top1_accuracy','hit_at_3','mrr','source']])
comparison.to_csv(OUT/'comparison.csv', index=False)
display(summary[['model','top1_correct','labelled_queries','top1_accuracy','hit_at_3','mrr','device','encode_all_seconds']])
display(details[['model','query','expected','top1','cosine_top1','top2','cosine_top2','target_rank']])
summary.to_csv(OUT/'summary.csv', index=False)
details.to_csv(OUT/'top3.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(score_rows).to_csv(OUT/'cosine-all.csv', index=False, encoding='utf-8-sig')
result = {'created_at': datetime.now(timezone.utc).isoformat(), 'environment': {
    'python': platform.python_version(), 'torch': torch.__version__,
    'sentence_transformers': importlib.metadata.version('sentence-transformers'),
    'transformers': importlib.metadata.version('transformers'), 'device': DEVICE},
    'fixture': PAYLOAD, 'instruction': TASK, 'query_prompt': QUERY_PROMPT, 'precision': 'FP32',
    'archived_e5_source': ARCHIVED_E5['source'], 'summary': summaries, 'rows': all_rows,
    'limits': '12 author-labelled probes only, 10 scored; not comparable to the larger local probe. No cosine threshold is calibrated. RD6 is unlabelled.'}
(OUT/'results.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print('DONE — kết quả:', str(OUT))


## Giới hạn

- Qwen được encode cả câu cùng instruction; passage chỉ là tên sản phẩm, theo hướng dẫn chính thức.
- Prefix E5 và Qwen khác nhau theo convention từng model; cosine giữa các model không cùng thang hiệu chỉnh.
- Chấm cùng một nhãn đích và tập ứng viên; chỉ so chất lượng thứ hạng, không so một ngưỡng cosine chung.
- RD6 không mặc định là Ri6; sản phẩm không có trong catalog không được ép coi top-1 là đáp án đúng.
- Không điều chỉnh instruction theo kết quả Qwen trong lượt này. Chưa đo trên tập độc lập.
- Tài liệu model: https://huggingface.co/Qwen/Qwen3-Embedding-0.6B
